# 数据重构代码

* 模型设计，训练，测试

* 重构后数据的转存--> 用于后续任务

*  这个代码用到了 FFT loss   L1 loss    以及corr loss    损失权重配置使用的可训练参数

In [ ]:
import torch
import torch.nn.functional as F

def frequency_loss(output, target):
    output_fft = torch.fft.fft(output, dim=-1)
    target_fft = torch.fft.fft(target, dim=-1)
    
    real_loss = F.mse_loss(output_fft.real, target_fft.real)
    imag_loss = F.mse_loss(output_fft.imag, target_fft.imag)
    
    return real_loss, imag_loss

def correlation_loss(output, target):

    output_mean = output.mean(dim=-1, keepdim=True)
    target_mean = target.mean(dim=-1, keepdim=True)

    output_centered = output - output_mean
    target_centered = target - target_mean

    numerator = (output_centered * target_centered).sum(dim=-1)
    denominator = (
        torch.sqrt((output_centered ** 2).sum(dim=-1)) * 
        torch.sqrt((target_centered ** 2).sum(dim=-1)) + 1e-8  
    )
    
    corr = numerator / denominator  
    corr_loss = 1 - corr.mean()     
    
    return corr_loss


class DynamicWeightedLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.log_sigma_time = torch.nn.Parameter(torch.tensor(0.0))
        self.log_sigma_real = torch.nn.Parameter(torch.tensor(0.0))
        self.log_sigma_imag = torch.nn.Parameter(torch.tensor(0.0))
        self.log_sigma_corr = torch.nn.Parameter(torch.tensor(0.0))

    def forward(self, output, target):
        time_loss = F.l1_loss(output, target)
        real_loss, imag_loss = frequency_loss(output, target)
        corr = correlation_loss(output, target)

        loss = (
            1 * torch.exp(-2 * self.log_sigma_time) * time_loss + self.log_sigma_time +
            0.1 * torch.exp(-2 * self.log_sigma_real) * real_loss + self.log_sigma_real +
            0.1 * torch.exp(-2 * self.log_sigma_imag) * imag_loss + self.log_sigma_imag +
            0.1 * torch.exp(-2 * self.log_sigma_corr) * corr + self.log_sigma_corr
        )

        return loss


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
import torch


train_data= np.load("train_data_small.npy")
print(train_data.shape)

X_train = train_data[:,0,:,:]
Y_train = train_data[:,1,:,:]


X = [np.roll(X_train[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
X_train = np.concatenate(X,axis=1)
Y = [np.roll(Y_train[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
Y_train = np.concatenate(Y,axis=1)
print(X_train.shape,Y_train.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_data = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from models import Multi_Chan_Conv,set_seed

class EarlyStopping:
    def __init__(self, patience=10, delta=1e-4):
        self.patience = patience
        self.delta = delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# ---------------------- 参数设置 ----------------------
batch_size = 32
num_epochs = 150
best_val_loss = float('inf')
best_model_path = "Multi_Chan_Conv_Corr_L1_FFT_norm_best.pth"
train_log = "Multi_Chan_Conv_Corr_L1_FFT_norm.txt"
early_stopping = EarlyStopping(patience=5)

set_seed(3407)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = Multi_Chan_Conv().to(device)
criterion = nn.L1Loss()
combined_loss = DynamicWeightedLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.002)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

with open(train_log, 'w',encoding='utf-8') as f:
    f.write("Epoch\tTrain_Loss\tVal_Loss\n")


for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, targets = inputs.to(device), targets.to(device)
        targets = targets.reshape(-1, 64, 550)

        optimizer.zero_grad()
        outputs = model(inputs)
        # loss = criterion(outputs, targets)
        # loss.backward()
        loss = combined_loss(outputs, targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() *inputs.shape[0]


    model.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            targets = targets.reshape(-1, 64, 550)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.shape[0]

    train_loss_avg = train_loss / len(train_data)
    val_loss_avg = val_loss / len(val_data)


    log_line = f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss_avg:.8f}, Val Loss: {val_loss_avg:.8f}"
    print(log_line)
    
    with open(train_log, 'a',encoding='utf-8') as f:

        f.write(f"{epoch+1}\t{train_loss_avg:.8f}\t{val_loss_avg:.8f}\n")

    # 学习率调度 + 早停判断
    scheduler.step(val_loss_avg)
    early_stopping.step(val_loss_avg)

    if val_loss_avg < best_val_loss:
        best_val_loss = val_loss_avg
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ Best model saved at epoch {epoch+1}, Val Loss: {val_loss_avg:.8f}")
        with open(train_log, 'a',encoding='utf-8') as f:
            f.write(f"Best model saved at epoch {epoch+1},\t Val Loss: {val_loss_avg:.8f}\n")

    # 判断早停
    if early_stopping.early_stop:
        print(f"⛔ Early stopping triggered at epoch {epoch+1}")
        with open(train_log, 'a',encoding='utf-8') as f:
            f.write(f"⛔ Early stopping triggered at epoch {epoch+1}")
        break


# ---------------------- 模型保存 ----------------------
torch.save(model.state_dict(), "Multi_Chan_Conv_Corr_L1_FFT_norm_final_1.pth")


In [ ]:
from models import Multi_Chan_Conv
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_data = np.load("test_data_small.npy")
X_test = test_data[:, 0, :, :]
Y_test = test_data[:, 1, :, :]

X = [np.roll(X_test[:, np.newaxis, i*8:i*8+8, :], -i, axis=-2) for i in range(8)]
X_test = np.concatenate(X, axis=1)
Y = [np.roll(Y_test[:, np.newaxis, i*8:i*8+8, :], -i, axis=-2) for i in range(8)]
Y_test = np.concatenate(Y, axis=1)
print("测试集 shape:", X_test.shape, Y_test.shape)

test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                              torch.tensor(Y_test, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model = Multi_Chan_Conv().to(device)
model.load_state_dict(torch.load("Multi_Chan_Conv_Corr_L1_FFT_norm_best_1.pth"))
model.eval()

criterion = nn.SmoothL1Loss()
criterion_L1 = nn.L1Loss()

test_loss = 0
test_L1_loss = 0
input_L1_loss = 0

X_mean = torch.Tensor(X_mean).to(device)
X_std = torch.Tensor(X_std).to(device)
Y_mean = torch.Tensor(Y_mean).to(device)
Y_std = torch.Tensor(Y_std).to(device)

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)

        outputs = outputs.reshape(outputs.shape[0],8,8,550)

        # 反归一化
        outputs_denorm = outputs * (Y_std + 1e-6) + Y_mean
        targets_denorm = targets * (Y_std + 1e-6) + Y_mean
        inputs_denorm = inputs * (X_std + 1e-6) + X_mean

        outputs_denorm = outputs_denorm.reshape(-1, 64, 550)
        targets_denorm = targets_denorm.reshape(-1, 64, 550)
        inputs_denorm = inputs_denorm.reshape(-1, 64, 550)

        test_loss += criterion(outputs_denorm, targets_denorm).item() * inputs.shape[0]
        input_L1_loss += criterion_L1(inputs_denorm, targets_denorm).item() * inputs.shape[0]
        test_L1_loss += criterion_L1(outputs_denorm, targets_denorm).item() * inputs.shape[0]

# ---------- 🔹 打印结果 ----------
print(f"✅ Test Total Loss (SmoothL1): {test_loss / len(test_dataset):.10f}")
print(f"✅ Test Input L1 Loss: {input_L1_loss / len(test_dataset):.6f}")
print(f"✅ Test Pred  L1 Loss: {test_L1_loss / len(test_dataset):.6f}")


## 将有噪声的训练数据和测试数据进行重构



In [ ]:
import os
import scipy
from scipy import io
import numpy as np
import torch

# 获取指定路径下的所有文件名
path= "noise_signal/damage1/"
path1= "origin_signal/damage1/"

file_names = os.listdir(path)
# print(file_names)
print(len(file_names))

reload_model = Multi_Chan_Conv()
reload_model.load_state_dict(torch.load("chonggou_model_pth/Multi_Chan_Conv_Corr_L1_FFT_norm_best_1.pth"))
reload_model.eval()

# ---------- ✅ 加载训练集均值和方差 ----------
X_mean = torch.Tensor(np.load("FFT_L1loss_norms/X_mean.npy"))
X_std = torch.Tensor(np.load("FFT_L1loss_norms/X_std.npy"))
Y_mean = torch.Tensor(np.load("FFT_L1loss_norms/Y_mean.npy"))
Y_std =torch.Tensor(np.load("FFT_L1loss_norms/Y_std.npy"))

all_mses = []

num = 0

for name in file_names:
    # 在获取干扰数据的同时需要获取目标数据
    if name[0] !="x":
        continue
    
    # 读取 .mat 文件
    mat_data_x = io.loadmat(path+name)
    name_y = name[:name.rfind('_')]
    mat_data_y = io.loadmat(path1+name_y+'.mat')

    mat_data_x = np.array([mat_data_x["data1_noise"].T])
    mat_data_y = np.array([mat_data_y[name_y+"_1"].T])
    
    # print(mat_data_x.shape,mat_data_y.shape)
    
    X = [np.roll(mat_data_x[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
    
    X = torch.tensor(np.concatenate(X,axis=1),dtype=torch.float32)
    
    Y = [np.roll(mat_data_y[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
    
    Y =  torch.tensor(np.concatenate(Y,axis=1),dtype=torch.float32)
    
    # ---------- ✅ 测试集归一化 ----------
    X_norm = (X - X_mean) / (X_std + 1e-6)
    
    # print(X_norm)


    cg_data = reload_model(X_norm)
    cg_data = cg_data.reshape(cg_data.shape[0],8,8,550)
    
    # 反归一化
    cg_data_denorm = cg_data * (Y_std + 1e-6) + Y_mean

    inputs_denorm = X_norm * (X_std + 1e-6) + X_mean
    
    
    mse = torch.mean((cg_data_denorm-Y)**2)
    
    # print(mse.item())
    all_mses.append(mse.item())
    
    # 重构数据的保存
    
    np.save(f"重构后数据用于训练/data_1/damage1/{name}.npy",cg_data_denorm.squeeze().detach().numpy())  
    
    num +=1 
    
    if num // 1000 ==0:
        print("进度：",num)
    
    
len(all_mses)

In [ ]:
print(sum(all_mses)/len(all_mses))

In [ ]:
import os
import scipy
from scipy import io
import numpy as np
import torch

# 获取指定路径下的所有文件名
path= "noise_signal/damage2/"
path1= "origin_signal/damage2/"

file_names = os.listdir(path)
# print(file_names)
print(len(file_names))

reload_model = Multi_Chan_Conv()
reload_model.load_state_dict(torch.load("chonggou_model_pth/Multi_Chan_Conv_Corr_L1_FFT_norm_best_1.pth"))
reload_model.eval()

# ---------- ✅ 加载训练集均值和方差 ----------
X_mean = torch.Tensor(np.load("FFT_L1loss_norms/X_mean.npy"))
X_std = torch.Tensor(np.load("FFT_L1loss_norms/X_std.npy"))
Y_mean = torch.Tensor(np.load("FFT_L1loss_norms/Y_mean.npy"))
Y_std =torch.Tensor(np.load("FFT_L1loss_norms/Y_std.npy"))

all_mses = []

num = 0

for name in file_names:
    # 在获取干扰数据的同时需要获取目标数据
    if name[0] !="x":
        continue
    
    # 读取 .mat 文件
    mat_data_x = io.loadmat(path+name)
    name_y = name[:name.rfind('_')]
    mat_data_y = io.loadmat(path1+name_y+'.mat')

    mat_data_x = np.array([mat_data_x["data1_noise"].T])
    mat_data_y = np.array([mat_data_y[name_y+"_1"].T])
    
    # print(mat_data_x.shape,mat_data_y.shape)
    
    X = [np.roll(mat_data_x[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
    
    X = torch.tensor(np.concatenate(X,axis=1),dtype=torch.float32)
    
    Y = [np.roll(mat_data_y[:,np.newaxis,i*8:i*8+8,:],-i,axis=-2) for i in range(0,8)]
    
    Y =  torch.tensor(np.concatenate(Y,axis=1),dtype=torch.float32)
    
    # ---------- ✅ 测试集归一化 ----------
    X_norm = (X - X_mean) / (X_std + 1e-6)
    
    # print(X_norm)


    cg_data = reload_model(X_norm)
    cg_data = cg_data.reshape(cg_data.shape[0],8,8,550)
    
    # 反归一化
    cg_data_denorm = cg_data * (Y_std + 1e-6) + Y_mean

    inputs_denorm = X_norm * (X_std + 1e-6) + X_mean
    
    
    mse = torch.mean((cg_data_denorm-Y)**2)
    
    # print(mse.item())
    all_mses.append(mse.item())
    
    # 重构数据的保存
    
    np.save(f"重构后数据用于训练/data_1/damage2/{name}.npy",cg_data_denorm.squeeze().detach().numpy())  
    
    num +=1 
    
    if num % 500 ==0:
        print("进度：",num)
    # break 
    # train_data.append(np.array([mat_data_x[name[:-4]].T, mat_data_y["field_value2"].T]))
    
    
len(all_mses)

In [ ]:
print(sum(all_mses)/len(all_mses))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


y = np.load('重构后数据用于训练/data_1/damage1/x100_100_1.mat.npy')
y = y.reshape(-1,550)


x = np.arange(550)  # 假设所有数组长度相同

# 绘制每个数组

plt.plot(x, y[0,:], label=f'Line')
    
# 添加图例、标题和轴标签
plt.legend()
plt.title('Plot of Multiple 1D Arrays')
plt.xlabel('Index')
plt.ylabel('Value')

# 显示图形
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import io

y = io.loadmat('origin_signal/damage1/x100_100.mat')

# print(y)
y = np.array([y["x100_100_1"].T])
y = y.reshape(-1,550)


x = np.arange(550)  # 假设所有数组长度相同

# 绘制每个数组

plt.plot(x, y[0,:], label=f'Line')
    
# 添加图例、标题和轴标签
plt.legend()
plt.title('Plot of Multiple 1D Arrays')
plt.xlabel('Index')
plt.ylabel('Value')

# 显示图形
plt.show()